# Dasar teks: kenali data, bersihkan, kata dasar (stemming), stopword, kata khas per kelas

**Butuh:** tabel `train` dengan kolom teks (+ kolom label untuk bagian 4).
**Menghasilkan:** kolom `clean`, `stem`, laporan panjang teks, kata khas per kelas.

Bersihkan teks hanya untuk model berbasis kata (TF-IDF). Model BERT / sentence-transformers
lebih baik diberi teks **asli** (hanya buang URL/username), karena tanda baca dan imbuhan
juga informasi baginya.

## Cara menempel ke notebook-mu

| sel | isi | wajib? |
|---|---|---|
| 1 | lihat data: panjang teks, label, teks kembar, teks kembar dengan label beda | **ya** |
| 2 | `clean_text`: huruf kecil, buang URL/username/angka/simbol, kata gaul | TF-IDF |
| 3 | stemming Sastrawi (`makanannya` → `makan`) + buang stopword | opsional, coba dua-duanya |
| 4 | kata/frasa paling khas per kelas (chi²) | EDA, presentasi |
| 5 | train ↔ test: `clean`/`stem` ke test, kembar, hampir kembar, adversarial | **ya** |

Regex, ambil field, angka/tanggal dari teks: `umum/teks_regex.ipynb`.

## 0. Data contoh (lewati saat lomba)
Tweet berlabel emosi (anger, fear, happy, love, sadness).

In [ ]:
import pandas as pd

DEMO_FOLDER = "data_latihan"   # folder data_latihan (di sebelah repo)

train = pd.read_csv(f"{DEMO_FOLDER}/text/classification/emot_train.csv")
test = pd.read_csv(f"{DEMO_FOLDER}/text/classification/emot_test.csv")
print(train.head())

## 1. Lihat data
Teks sangat panjang (> 512 token) → BERT memotongnya; teks kembar dengan label beda = label
berisik (pilih label terbanyak atau buang).

In [ ]:
TEXT = "text"     # UBAH
TARGET = "label"  # UBAH

lengths = train[TEXT].str.split().str.len()
print("jumlah kata per teks:\n", lengths.describe().round(1))
print(train[TARGET].value_counts())
print("teks kosong:", train[TEXT].isna().sum(), "| teks kembar:", train[TEXT].duplicated().sum())
labels_per_text = train.groupby(TEXT)[TARGET].nunique()
print("teks kembar dengan label berbeda:", (labels_per_text > 1).sum())
for label in sorted(train[TARGET].unique()):
    print(f"[{label}]", train.loc[train[TARGET] == label, TEXT].iloc[0][:100])

## 2. `clean_text` → kolom `clean`
`SLANG`: kamus kata gaul → baku (tulis sendiri dari kata yang sering muncul di bagian 4).

In [ ]:
import re

SLANG = {"gak": "tidak", "ga": "tidak", "nggak": "tidak", "tdk": "tidak", "yg": "yang",
         "dgn": "dengan", "bgt": "banget", "udh": "sudah", "udah": "sudah", "aja": "saja",
         "tp": "tapi", "krn": "karena", "karna": "karena", "sy": "saya", "gue": "saya",
         "gw": "saya", "lu": "kamu", "lo": "kamu", "dr": "dari", "jd": "jadi"}   # UBAH


def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"\[username\]|@\w+", " ", text)          # username
    text = re.sub(r"https?://\S+|www\.\S+|\[url\]", " ", text)
    text = re.sub(r"#(\w+)", r"\1", text)                    # #hashtag → hashtag
    text = re.sub(r"(\w)\1{2,}", r"\1", text)                # bangeeettt → banget
    text = re.sub(r"[^a-z\s]", " ", text)                    # UBAH: angka dibuang
    words = []
    for word in text.split():
        words.append(SLANG.get(word, word))
    return " ".join(words)


print(clean_text("[USERNAME] Gak bangeeettt!!! cek https://x.id #kecewa 😡"))
train["clean"] = train[TEXT].apply(clean_text)
print(train[["clean"]].head(3))

## 3. Stemming (kata dasar) + buang stopword → kolom `stem`
Sastrawi lambat (±1 ms/kata) → setiap kata unik di-stem SEKALI lalu disimpan di dict.
Stopword bawaan Sastrawi membuang `tidak`/`bukan` juga; untuk sentimen, keluarkan dari daftar.

In [ ]:
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from tqdm import tqdm

stemmer = StemmerFactory().create_stemmer()
STOPWORDS = set(StopWordRemoverFactory().get_stop_words())
for keep in ["tidak", "bukan", "belum", "jangan", "kurang"]:   # UBAH: kata penting (sentimen)
    STOPWORDS.discard(keep)

unique_words = set()
for text in train["clean"]:
    unique_words.update(text.split())
stem_of = {}
for word in tqdm(unique_words):
    stem_of[word] = stemmer.stem(word)


def stem_text(text):
    words = []
    for word in text.split():
        if word not in STOPWORDS:
            words.append(stem_of.get(word, word))
    return " ".join(words)


train["stem"] = train["clean"].apply(stem_text)
print(train["clean"].iloc[1][:90], "\n→", train["stem"].iloc[1][:90])

## 4. Kata/frasa paling khas per kelas (chi²)
Chi² tinggi = kata itu muncul jauh lebih sering di satu kelas dibanding kelas lain.
Berguna untuk: mengerti data, menambah kamus `SLANG`, bahan presentasi.

In [ ]:
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_selection import chi2

TOP_WORDS = 10   # UBAH

counter = CountVectorizer(ngram_range=(1, 2), min_df=3)
counts = counter.fit_transform(train["clean"])
words = np.array(counter.get_feature_names_out())
for label in sorted(train[TARGET].unique()):
    in_class = (train[TARGET] == label).to_numpy()
    scores, _ = chi2(counts, in_class)
    scores = np.nan_to_num(scores)
    rate_in = np.asarray(counts[in_class].mean(axis=0)).ravel()
    rate_out = np.asarray(counts[~in_class].mean(axis=0)).ravel()
    scores[rate_in <= rate_out] = 0        # buang kata yang justru JARANG di kelas ini
    print(f"{label}: {', '.join(words[np.argsort(-scores)[:TOP_WORDS]])}")

## 5. Train ↔ test: apakah test mirip train?
Soal sering menyimpan jebakan di sini: teks test yang SAMA dengan train (jawabannya sudah
diketahui), yang hampir sama (hanya beda tanda baca/huruf), atau test yang gayanya lain (domain
beda → skor validasi menipu). Urutan: terapkan pembersihan yang sama ke test, lalu cek tiga hal.

In [ ]:
from tqdm import tqdm

TEST_TEXT = "text"   # UBAH: kolom teks di test

raw_test = test[TEST_TEXT].fillna("").astype(str)
test["clean"] = raw_test.apply(clean_text)
unseen = set()
for text in test["clean"]:
    unseen.update(text.split())
new_words = unseen - set(stem_of)
for word in tqdm(new_words):
    stem_of[word] = stemmer.stem(word)      # kata baru di test di-stem juga
test["stem"] = test["clean"].apply(stem_text)
print(test[["clean", "stem"]].head(3))
print("kata unik test:", len(unseen), "| belum pernah muncul di train:", len(new_words))

### 5a. Kembar persis dan hampir kembar
Persis: `clean` test sama dengan `clean` train (label train = tebakan hampir pasti benar, tetapi
kalau label train sendiri berisik, itu ikut). Hampir kembar: kemiripan kosinus TF-IDF huruf
≥ `SIMILAR_LEVEL` (0,8 longgar, 0,95 ketat).

In [ ]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors

SIMILAR_LEVEL = 0.9   # UBAH


def top_label(labels):
    return labels.value_counts().index[0]


label_of = train.groupby("clean")[TARGET].agg(top_label)
exact = test["clean"].isin(label_of.index)
print("test persis ada di train:", exact.sum(), "dari", len(test), f"({exact.mean():.1%})")
print("test kembar di dalam test sendiri:", test["clean"].duplicated().sum())
chars = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=2, sublinear_tf=True)
train_chars = chars.fit_transform(train["clean"])
sample = test["clean"].head(5000)
near = NearestNeighbors(n_neighbors=1, metric="cosine").fit(train_chars)
distances, nearest = near.kneighbors(chars.transform(sample))
similarity = 1 - distances[:, 0]
close = np.where((similarity >= SIMILAR_LEVEL) & (similarity < 0.999))[0]
print(f"hampir kembar (sim >= {SIMILAR_LEVEL}, bukan persis):", len(close), "dari", len(sample))
for row in close[:3]:
    print(round(similarity[row], 3), "|", sample.iloc[row][:50], "|",
          train["clean"].iloc[nearest[row, 0]][:50], "|", train[TARGET].iloc[nearest[row, 0]])

### 5b. Adversarial validation + panjang teks + kata baru
Latih model "ini baris test atau train?" dengan CV. AUC ≈ 0,5 = test sejenis train. AUC > 0,65 =
test beda (domain/gaya/periode): validasi hati-hati, pertimbangkan pseudo-label atau split yang
meniru test. Kata paling khas test menunjukkan bedanya apa.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.pipeline import make_pipeline

both = pd.concat([train["clean"], test["clean"]], ignore_index=True)
is_test = np.r_[np.zeros(len(train)), np.ones(len(test))]
pipe = make_pipeline(TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True),
                     LogisticRegression(C=1, max_iter=500))
folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
proba = cross_val_predict(pipe, both, is_test, cv=folds, method="predict_proba")
auc = roc_auc_score(is_test, proba[:, 1])
print("adversarial AUC:", round(auc, 3), "(0.5 = sama, > 0.65 = beda)")
pipe.fit(both, is_test)
words = np.array(pipe[0].get_feature_names_out())
order = np.argsort(pipe[-1].coef_[0])
print("paling khas TEST :", ", ".join(words[order[-8:]]))
print("paling khas TRAIN:", ", ".join(words[order[:8]]))
for name, table in [("train", train), ("test", test)]:
    length = table["clean"].str.split().str.len()
    print(name, "kata per teks: median", length.median(), "| p95", length.quantile(0.95))